In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import os
import json
import csv
import hashlib
import pandas as pd
from pathlib import Path
from collections import defaultdict

ROOT = Path("/kaggle/input")
MAX_PREVIEW = 1200
MAX_HASH_SIZE = 50 * 1024 * 1024

files_report = []
content_report = []
hashes = defaultdict(list)

def inspect_file(path):
    size = path.stat().st_size
    ext = path.suffix.lower()
    result = {
        "file": str(path.relative_to(ROOT)),
        "size_mb": round(size / 1024**2, 3),
        "type": ext or "unknown",
        "status": "OK",
        "details": ""
    }

    try:
        if size == 0:
            result["status"] = "EMPTY"
            return result

        if ext == ".csv":
            df = pd.read_csv(path, nrows=5)
            result["details"] = f"Columns: {list(df.columns)}"
            content_report.append((path.name, df.head(3).to_string()))

        elif ext in (".jsonl", ".ndjson"):
            with open(path, encoding="utf-8", errors="replace") as f:
                lines = [next(f) for _ in range(3)]
            result["details"] = "JSON Lines; sample records available"
            content_report.append((path.name, "\n".join(lines)[:MAX_PREVIEW]))

        elif ext == ".json":
            with open(path, encoding="utf-8", errors="replace") as f:
                sample = f.read(50000)
            try:
                data = json.loads(sample)
                result["details"] = f"JSON structure: {type(data).__name__}"
                content_report.append((path.name, json.dumps(data, ensure_ascii=False)[:MAX_PREVIEW]))
            except json.JSONDecodeError:
                result["details"] = "Could not parse first 50 KB as complete JSON"

        elif ext in (".parquet", ".pq"):
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(path)
            result["details"] = (
                f"Rows: {pf.metadata.num_rows}; "
                f"Columns: {pf.schema.names}"
            )
            content_report.append((path.name, str(pf.schema)[:MAX_PREVIEW]))

        elif ext in (".txt", ".md", ".py", ".js", ".ts", ".html", ".css", ".sql"):
            with open(path, encoding="utf-8", errors="replace") as f:
                sample = f.read(MAX_PREVIEW)
            result["details"] = "Text/source file"
            content_report.append((path.name, sample))

        elif ext in (".zip", ".gz", ".tar", ".7z"):
            result["details"] = "Compressed archive; contents not extracted"

        elif ext in (".bin", ".safetensors", ".gguf", ".pt", ".pth", ".onnx"):
            result["details"] = "Model or binary file; not inspected as training text"

        else:
            result["details"] = "File type needs manual inspection"

        # Hash only smaller files to find exact duplicates.
        if size <= MAX_HASH_SIZE:
            h = hashlib.sha256()
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    h.update(chunk)
            hashes[h.hexdigest()].append(str(path.relative_to(ROOT)))

    except Exception as e:
        result["status"] = "CHECK"
        result["details"] = str(e)[:250]

    return result


print("Scanning attached Kaggle datasets...")

if not ROOT.exists():
    print("ERROR: /kaggle/input does not exist.")
else:
    for path in ROOT.rglob("*"):
        if path.is_file():
            files_report.append(inspect_file(path))

report = pd.DataFrame(files_report)

print("\n========== DATASET FILE REPORT ==========")
if not report.empty:
    print(report.to_string(index=False))
else:
    print("No files found. Attach datasets using Add Input.")

print("\n========== POSSIBLE EXACT DUPLICATES ==========")
duplicate_groups = [v for v in hashes.values() if len(v) > 1]
if duplicate_groups:
    for group in duplicate_groups:
        print("\nDUPLICATE:", group)
else:
    print("No duplicates found among files hashed.")

print("\n========== CONTENT PREVIEWS ==========")
for name, preview in content_report[:100]:
    print(f"\n--- {name} ---")
    print(str(preview)[:MAX_PREVIEW])

# Save reports for download from Kaggle Output.
report.to_csv("/kaggle/working/bella_file_report.csv", index=False)

with open("/kaggle/working/bella_content_previews.txt", "w", encoding="utf-8") as f:
    for name, preview in content_report:
        f.write(f"\n\n===== {name} =====\n{preview}\n")

print("\n========== COMPLETE ==========")
print("Reports saved:")
print("/kaggle/working/bella_file_report.csv")
print("/kaggle/working/bella_content_previews.txt")


Scanning attached Kaggle datasets...

========== DATASET FILE REPORT ==========


In [4]:
from pathlib import Path

root = Path("/kaggle/input/datasets")

for publisher in sorted(root.iterdir()):
    if publisher.is_dir():
        datasets = [p.name for p in publisher.iterdir() if p.is_dir()]
        files = [p.name for p in publisher.iterdir() if p.is_file()]

        print(f"\n{publisher.name}")
        print("  Dataset folders:", datasets[:10])
        print("  Files:", files[:10])


aiw0nderland
  Dataset folders: ['bella-dataset-01']
  Files: []

bcsmilyai
  Dataset folders: ['openthoughts-formatted-for-llm']
  Files: []

deepakj100
  Dataset folders: ['swe-smith-golden-dataset']
  Files: []

godelev
  Dataset folders: ['53m-token-instruction-code-and-qa-dataset']
  Files: []

kaptaan45
  Dataset folders: ['kapcode-1b']
  Files: []

karstenkuhnke
  Dataset folders: ['mimo-v26-distill-qwen9b-webdev-chunk00']
  Files: []

kshitijftw
  Dataset folders: ['swe-smith-python-small']
  Files: []

kutayahin
  Dataset folders: ['stackoverflow-programming-questions-2020-2025']
  Files: []

lhawauthors
  Dataset folders: ['lhaw-benchmark']
  Files: []

meruvakodandasuraj
  Dataset folders: ['llm-benchmark-tracker-20242026']
  Files: []

minhduy1222
  Dataset folders: ['toolbench-agent']
  Files: []

muhammad4hmed
  Dataset folders: ['malwaremicrosoftbig']
  Files: []

ngwkhai
  Dataset folders: ['natural-language-to-project-structure']
  Files: []

nsaravana
  Dataset folde

In [5]:
from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/input/datasets")

rows = []

for publisher in sorted(ROOT.iterdir()):
    if not publisher.is_dir():
        continue

    for dataset in sorted(publisher.iterdir()):
        if not dataset.is_dir():
            continue

        for file in dataset.rglob("*"):
            if file.is_file():
                rows.append({
                    "publisher": publisher.name,
                    "dataset": dataset.name,
                    "file": file.name,
                    "extension": file.suffix.lower(),
                    "size_mb": round(file.stat().st_size / (1024 * 1024), 2),
                    "path": str(file)
                })

report = pd.DataFrame(rows)

print("DATASETS:", report["dataset"].nunique() if not report.empty else 0)
print("FILES:", len(report))
display(report)

output = "/kaggle/working/bella_file_report.csv"
report.to_csv(output, index=False)
print("SAVED:", output)

DATASETS: 27
FILES: 74683


,publisher,dataset,file,extension,size_mb,path
0,aiw0nderland,bella-dataset-01,README.md,.md,0.01,/kaggle/input/datasets/aiw0nderland/bella-data...
1,aiw0nderland,bella-dataset-01,test-00000-of-00001.parquet,.parquet,0.76,/kaggle/input/datasets/aiw0nderland/bella-data...
2,aiw0nderland,bella-dataset-01,.gitattributes,,0.00,/kaggle/input/datasets/aiw0nderland/bella-data...
3,aiw0nderland,bella-dataset-01,dataset.json,.json,0.45,/kaggle/input/datasets/aiw0nderland/bella-data...
4,aiw0nderland,bella-dataset-01,config,,0.00,/kaggle/input/datasets/aiw0nderland/bella-data...
...,...,...,...,...,...,...
74678,verracodeguacas,swe-smith-trajectories-transformed,meta.jsonl,.jsonl,0.34,/kaggle/input/datasets/verracodeguacas/swe-smi...
74679,wukeneth,cct-data,openhermes_40k.json,.json,76.63,/kaggle/input/datasets/wukeneth/cct-data/openh...
74680,wukeneth,openthoughts-114k-qwen3,openthoughts_114k_train.jsonl,.jsonl,2359.44,/kaggle/input/datasets/wukeneth/openthoughts-1...
74681,wukeneth,openthoughts-114k-qwen3,openthoughts_114k_eval.jsonl,.jsonl,86.74,/kaggle/input/datasets/wukeneth/openthoughts-1...


SAVED: /kaggle/working/bella_file_report.csv


In [6]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/kaggle/input/datasets")
results = []

for file in ROOT.rglob("*"):
    if not file.is_file():
        continue

    ext = file.suffix.lower()

    try:
        if ext == ".jsonl":
            with open(file, encoding="utf-8") as f:
                samples = [json.loads(line) for _, line in zip(range(2), f) if line.strip()]
            preview = str(samples)[:1200]

        elif ext == ".parquet":
            df = pd.read_parquet(file)
            preview = str({
                "columns": list(df.columns),
                "samples": df.head(2).to_dict("records")
            })[:1200]

        elif ext == ".json":
            with open(file, encoding="utf-8") as f:
                data = json.load(f)
            preview = str(data)[:1200]

        else:
            continue

        results.append({
            "dataset": file.parent.name,
            "file": file.name,
            "format": ext,
            "preview": preview
        })

    except Exception as e:
        results.append({
            "dataset": file.parent.name,
            "file": file.name,
            "format": ext,
            "preview": "ERROR: " + str(e)[:300]
        })

content_report = pd.DataFrame(results)

print("INSPECTED FILES:", len(content_report))
display(content_report)
content_report.to_csv("/kaggle/working/bella_content_report.csv", index=False)
print("SAVED: /kaggle/working/bella_content_report.csv")

INSPECTED FILES: 54090


,dataset,file,format,preview
0,mimo-v26-distill-qwen9b-webdev-chunk00,GEN_RESULT.json,.json,"{'phases': {'G1_install': {'status': 'OK', 'se..."
1,mimo-v26-distill-qwen9b-webdev-chunk00,train.jsonl,.jsonl,"[{'task_id': 'dasyn_260630_00578', 'prompt': '..."
2,kapcode-1b,processing_report.json,.json,{'final_stats': {'output_dir': '/kaggle/workin...
3,kapcode-1b,manifest.json,.json,"{'created_at': '2026-08-14T12:08:11Z', 'output..."
4,openthoughts-114k-qwen3,openthoughts_114k_train.jsonl,.jsonl,"[{'conversations': [{'from': 'human', 'value':..."
...,...,...,...,...
54085,G1_answer,62235_ChatGPT_DFS_woFilter_w2.json,.json,"{'win': True, 'tree': {'size': 7, 'max_length'..."
54086,G1_answer,34576_ChatGPT_DFS_woFilter_w2.json,.json,"{'win': True, 'tree': {'size': 25, 'max_length..."
54087,G1_answer,866_ChatGPT_DFS_woFilter_w2.json,.json,"{'win': True, 'tree': {'size': 80, 'max_length..."
54088,G1_answer,81283_ChatGPT_DFS_woFilter_w2.json,.json,"{'win': True, 'tree': {'size': 26, 'max_length..."


SAVED: /kaggle/working/bella_content_report.csv


In [9]:
!pwd
!git status
!git remote -v
!git branch --show-current

/kaggle/working
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [10]:
!ls -la /kaggle/working
!ls -la /kaggle/input
!find /kaggle/working -maxdepth 3 -name .git -type d

total 139724
drwxr-xr-x 3 root root     4096 Nov  2 02:35 .
drwxr-xr-x 5 root root     4096 Nov  2 02:08 ..
-rw-r--r-- 1 root root 64555616 Nov  2 02:18 bella_content_previews.txt
-rw-r--r-- 1 root root 63285125 Nov  2 02:35 bella_content_report.csv
-rw-r--r-- 1 root root 15217759 Nov  2 02:27 bella_file_report.csv
drwxr-xr-x 2 root root     4096 Nov  2 02:09 .virtual_documents
total 12
drwxr-xr-x  3 root root 4096 Nov  2 02:02 .
drwxr-xr-x  5 root root 4096 Nov  2 02:08 ..
drwxr-xr-x 28 root root 4096 Nov  2 02:08 datasets
